# PixelRNN (Row LSTM) — autoregressive images with recurrence

> Tutorial pair for [`pixelrnn.py`](pixelrnn.py).

## 1. Intuition
PixelRNN shares PixelCNN's promise -- model an image pixel by pixel in raster
order and get an exact likelihood -- but uses an **LSTM** to summarize the context
instead of a fixed-size masked convolution. The recurrence carries information
down the rows, so in principle a pixel can depend on *all* the rows above it, not
just a small convolutional window. This file implements the simpler **Row LSTM**
variant: scan top to bottom, one LSTM step per row.

## 2. Concept (the slide)
- Same chain-rule factorization over pixels as PixelCNN.
- **Row LSTM:** an LSTM whose hidden state moves down the image one row at a time.
- The input to row $i$ is a *causal 1-D convolution* of the row above, and the
  recurrence uses row $i-1$ to predict row $i$ (a one-row shift) -- so a pixel
  never sees its own row, preserving causality.
- Train by maximum likelihood with teacher forcing; sample pixel by pixel.

## 3. Math derivation — factorization & the row recurrence

**Autoregressive factorization** (identical chain rule as PixelCNN). With the
$N=H\cdot W$ pixels in raster order,
$$\boxed{\,p(x)=\prod_{i=1}^{N}p\big(x_i\mid x_{<i}\big)\,},
\qquad p(x_i\mid x_{<i})=\sigma(\ell_i)^{x_i}(1-\sigma(\ell_i))^{1-x_i}.$$
The exact NLL is again a sum of per-pixel binary cross-entropies.

**Row LSTM recurrence.** Group the conditionals by row. Let $r_i\in\{0,1\}^W$ be
row $i$. We compute a state that carries the context of rows $<i$ downward:
$$\text{input}_i=\mathrm{Conv1d}^{\text{causal}}_{\text{cols}}\big(\phi(r_{i-1})\big),
\qquad (h_i,c_i)=\mathrm{LSTM}\big(\text{input}_i,(h_{i-1},c_{i-1})\big),$$
$$\ell_i=\mathrm{readout}(h_i)\in\mathbb R^{W}.$$
Two design points guarantee the autoregressive property:
1. **One-row shift:** row $i$ is predicted from $r_{i-1}$ and the recurrent state,
   which only ever absorbed rows $\le i-1$. So $x_i$ never enters its own
   prediction.
2. **Causal 1-D conv** along the width: with kernel size $k$ the convolution mixes
   a few neighbouring columns of the row above, giving each pixel a *triangular*
   receptive field that widens with depth in the rows -- the LSTM extends it
   unboundedly far up the image.

The recurrence is exactly an RNN unrolled over $H$ steps, so training uses
**backpropagation through time** (with gradient clipping, since BPTT can explode --
the same issue showcased in `dl/rnn/`).

**Why recurrence over convolution?** A masked conv has a bounded receptive field;
the LSTM's state can in principle propagate dependencies across the *entire* image
height, at the cost of an inherently sequential forward pass.

## 4. Model — the Row-LSTM PixelRNN

In [ ]:
# ===== actual implementation from pixelrnn.py =====
from __future__ import annotations

import numpy as np

SEED = 0

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import load_digits
    X = load_digits().data.reshape(-1, 1, 8, 8) / 16.0
    X = (X > 0.3).astype(np.float32)

    m = RowLSTM(size=8, hidden=64).fit(X, epochs=35)
    nll = m.history[-1]
    print(f"PixelRNN (Row LSTM) final NLL = {nll:.2f} nats/image "
          f"({nll / 64:.3f} nats/pixel)")

    s = m.sample(16)
    print(f"  sampled {s.shape[0]} images, mean on-pixels="
          f"{s.mean():.3f} (data {X.mean():.3f})")


class RowLSTM(nn.Module):
    r"""
    Simplified Row-LSTM PixelRNN for binary 1x8x8 images.

    For each output row i we form an input feature by a *causal* 1-D convolution
    across columns of the embedded pixels of row i-1 (so a pixel sees its
    upper-left/up/upper-right neighbours from earlier rows, never its own row).
    A per-column LSTM then carries information down the rows.
    """

    def __init__(self, size: int = 8, hidden: int = 64, k: int = 3):
        super().__init__()
        self.size, self.hidden, self.pad = size, hidden, k // 2
        self.embed = nn.Conv2d(1, hidden, kernel_size=1)          # per-pixel embed
        # causal 1-D conv along the row (width dim); pad symmetrically (centered)
        self.row_conv = nn.Conv1d(hidden, hidden, k, padding=k // 2)
        self.cell = nn.LSTMCell(hidden, hidden)
        self.readout = nn.Linear(hidden, size)                    # logits for a row

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (N,1,H,W) -> per-pixel logits (N,1,H,W)."""
        n, _, H, W = x.shape
        dev = x.device
        feat = self.embed(x)                                      # (N, hidden, H, W)
        h = torch.zeros(n, self.hidden, device=dev)
        c = torch.zeros(n, self.hidden, device=dev)
        logits_rows = []
        # row 0 is predicted from an all-zero context (no rows above)
        prev_row_feat = torch.zeros(n, self.hidden, W, device=dev)
        for i in range(H):
            conv = self.row_conv(prev_row_feat)                   # (N, hidden, W)
            inp = conv.mean(dim=2)                                # summarize the row
            h, c = self.cell(inp, (h, c))                         # carry down rows
            logits_rows.append(self.readout(h))                   # (N, W) logits
            prev_row_feat = feat[:, :, i, :]                      # shift: use row i for row i+1
        logits = torch.stack(logits_rows, dim=1).unsqueeze(1)     # (N,1,H,W)
        return logits

    def loss(self, x: torch.Tensor) -> torch.Tensor:
        logits = self(x)
        return F.binary_cross_entropy_with_logits(logits, x, reduction="none") \
            .sum(dim=[1, 2, 3]).mean()

    def fit(self, X, epochs: int = 40, batch: int = 128, lr: float = 3e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.parameters(), 5.0)  # BPTT can explode
                opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int):
        """Ancestral sampling: fill the image row by row, pixel by pixel."""
        dev = next(self.parameters()).device
        H = W = self.size
        x = torch.zeros(n, 1, H, W, device=dev)
        for i in range(H):
            for j in range(W):
                logits = self(x)
                p = torch.sigmoid(logits[:, :, i, j])
                x[:, :, i, j] = torch.bernoulli(p)
        return x.cpu().numpy()

## 5. Training / sampling — NLL loss (BPTT + clipping) and raster sampler

In [ ]:
# ===== actual implementation from pixelrnn.py =====
class RowLSTM(nn.Module):
    r"""
    Simplified Row-LSTM PixelRNN for binary 1x8x8 images.

    For each output row i we form an input feature by a *causal* 1-D convolution
    across columns of the embedded pixels of row i-1 (so a pixel sees its
    upper-left/up/upper-right neighbours from earlier rows, never its own row).
    A per-column LSTM then carries information down the rows.
    """

    def __init__(self, size: int = 8, hidden: int = 64, k: int = 3):
        super().__init__()
        self.size, self.hidden, self.pad = size, hidden, k // 2
        self.embed = nn.Conv2d(1, hidden, kernel_size=1)          # per-pixel embed
        # causal 1-D conv along the row (width dim); pad symmetrically (centered)
        self.row_conv = nn.Conv1d(hidden, hidden, k, padding=k // 2)
        self.cell = nn.LSTMCell(hidden, hidden)
        self.readout = nn.Linear(hidden, size)                    # logits for a row

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (N,1,H,W) -> per-pixel logits (N,1,H,W)."""
        n, _, H, W = x.shape
        dev = x.device
        feat = self.embed(x)                                      # (N, hidden, H, W)
        h = torch.zeros(n, self.hidden, device=dev)
        c = torch.zeros(n, self.hidden, device=dev)
        logits_rows = []
        # row 0 is predicted from an all-zero context (no rows above)
        prev_row_feat = torch.zeros(n, self.hidden, W, device=dev)
        for i in range(H):
            conv = self.row_conv(prev_row_feat)                   # (N, hidden, W)
            inp = conv.mean(dim=2)                                # summarize the row
            h, c = self.cell(inp, (h, c))                         # carry down rows
            logits_rows.append(self.readout(h))                   # (N, W) logits
            prev_row_feat = feat[:, :, i, :]                      # shift: use row i for row i+1
        logits = torch.stack(logits_rows, dim=1).unsqueeze(1)     # (N,1,H,W)
        return logits

    def loss(self, x: torch.Tensor) -> torch.Tensor:
        logits = self(x)
        return F.binary_cross_entropy_with_logits(logits, x, reduction="none") \
            .sum(dim=[1, 2, 3]).mean()

    def fit(self, X, epochs: int = 40, batch: int = 128, lr: float = 3e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.parameters(), 5.0)  # BPTT can explode
                opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int):
        """Ancestral sampling: fill the image row by row, pixel by pixel."""
        dev = next(self.parameters()).device
        H = W = self.size
        x = torch.zeros(n, 1, H, W, device=dev)
        for i in range(H):
            for j in range(W):
                logits = self(x)
                p = torch.sigmoid(logits[:, :, i, j])
                x[:, :, i, j] = torch.bernoulli(p)
        return x.cpu().numpy()

## 6. Train & sample on binarized 8×8 digits

In [ ]:
demo()

## 7. Visualization — training NLL and generated digits

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
import pixelrnn as M

X = load_digits().data.reshape(-1, 1, 8, 8) / 16.0
X = (X > 0.3).astype("float32")
m = M.RowLSTM(size=8, hidden=64).fit(X, epochs=35)
samples = m.sample(16)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(m.history)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("NLL (nats/image)")
ax[0].set_title("Row-LSTM training likelihood")

# show samples in an inset grid
ax[1].axis("off"); ax[1].set_title("Generated digits (pixel by pixel)")
grid = np.zeros((2 * 8, 8 * 8))
for idx in range(16):
    r, c = divmod(idx, 8)
    grid[r * 8:(r + 1) * 8, c * 8:(c + 1) * 8] = samples[idx, 0]
ax[1].imshow(grid, cmap="gray")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- PixelRNN gives an **exact** likelihood, like PixelCNN, but with an unbounded
  vertical receptive field via the LSTM state.
- The **one-row shift + causal conv** is what keeps it autoregressive; get the
  shift wrong and the model trivially copies the target.
- Recurrence makes both training and sampling sequential -- slower than PixelCNN,
  which is why the field moved toward convolutional / transformer autoregressive
  models. **BPTT** here needs gradient clipping (cf. `dl/rnn/`).
- The full paper's **Diagonal BiLSTM** removes the Row-LSTM's blind spot; we
  implement only the simpler Row LSTM.